In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from curves import YieldCurve, load_treasury

# Initialize the client with your custom FRED API key
# Replace 'YOUR_API_KEY_HERE' with the actual key string from your account
API_KEY = '424034afb86bdcab88017369d65d00da'

df_treasury = load_treasury(API_KEY)
latest_rates = df_treasury.iloc[-1]
print("\nLatest Treasury Rates (Par Yields)(%):")
print(latest_rates)


In [ ]:
latest_rates = df_treasury.iloc[-1]
curve = YieldCurve(latest_rates, "linear")
print(curve.quotes)
curve.plot()


In [ ]:
curve = YieldCurve(latest_rates, "cubic_spline")
curve.plot()


The Nelson-Siegel curve is fitted to the continuously compounded yields:

$$
y(\tau)=\beta_0+\beta_1\frac{1-e^{-\tau/\lambda}}{\tau/\lambda}+\beta_2\left(\frac{1-e^{-\tau/\lambda}}{\tau/\lambda}-e^{-\tau/\lambda}\right)
$$

where $\tau$ is maturity in years. $\beta_0$ is the long-run level, $\beta_1$ is how far the short end sits from that level, $\beta_2$ is the hump, and $\lambda$ sets how fast the curve settles. Those four numbers are fitted to the eleven quoted yields.

Unlike the straight-line and cubic-spline charts, this curve does not pass through every quote. It is one smooth function, so a bend such as the dip from 20 years to 30 years can sit a few basis points off the dots.

In [ ]:
curve = YieldCurve(latest_rates, "nelson_siegel")
curve.plot()


Svensson adds a second hump to Nelson-Siegel:

$$
y(\tau)=\beta_0+\beta_1\frac{1-e^{-\tau/\lambda_1}}{\tau/\lambda_1}+\beta_2\left(\frac{1-e^{-\tau/\lambda_1}}{\tau/\lambda_1}-e^{-\tau/\lambda_1}\right)+\beta_3\left(\frac{1-e^{-\tau/\lambda_2}}{\tau/\lambda_2}-e^{-\tau/\lambda_2}\right)
$$

$\beta_3$ and $\lambda_2$ are that extra hump and its maturity. The fit tries a few starting points and keeps the one that is not stuck on a bound. Like Nelson-Siegel, the curve is one smooth function, so it does not pass through every quote. The second hump lets it follow a bend such as 20 years to 30 years more closely.

In [ ]:
curve = YieldCurve(latest_rates, "nss")
curve.plot()


The curve is a Matérn function whose length scale and noise are learned from the eleven quotes. A small noise term means it is not forced through every point. The gaps from 10 to 20 years and from 20 to 30 years are where the data say less about the yield in between.


The Gaussian process does not take arbitrage into account.

A discount curve is free of static arbitrage only if the discount factor falls as maturity gets longer. With continuous compounding that discount factor is $e^{-y(\tau)\tau}$, and it falls only when the instantaneous forward stays non-negative:

$$
f(\tau)=y(\tau)+\tau\,y'(\tau)\ge 0
$$

The fit never checks that. It only learns a smooth curve and a noise level from the eleven quotes, so paths within that noise can have forwards that go negative.

On the latest quotes the fitted curve can still stay clear of that. That is a property of this dataset, not a constraint in the model.


In [ ]:
curve = YieldCurve(latest_rates, "gp")
curve.plot()
forward = curve.curves()["forward_pct"]
print(f"minimum instantaneous forward = {forward.min():.2f}%")


The no-arbitrage curve is built from a positive instantaneous forward. A latent function $g$ is a Gaussian process with a Matérn-$\tfrac{5}{2}$ kernel, and the forward is

$$
f(\tau)=\operatorname{softplus}(g(\tau))=\ln(1+e^{g(\tau)})>0.
$$

The continuously compounded zero rate is the average of that forward:

$$
y(\tau)=\frac{1}{\tau}\int_0^{\tau} f(s)\,ds.
$$

Because $f$ stays positive, the discount factor $e^{-y(\tau)\tau}$ falls as maturity increases. The latent curve is chosen by minimizing

$$
\sum_i \frac{\bigl(y(\tau_i)-y_i^{\mathrm{obs}}\bigr)^2}{\sigma^2}+\|w\|^2,
$$

where $y_i^{\mathrm{obs}}$ are the quoted yields, $\sigma=0.05$, and $w$ are the Gaussian-process weights.

In [ ]:
curve = YieldCurve(latest_rates, "no_arb_gp")
curve.plot()
forward = curve.curves()["forward_pct"]
print(f"minimum instantaneous forward = {forward.min():.2f}%")


A zero curve allows static arbitrage if its discount factor ever rises with maturity. That is the same as the instantaneous forward going negative:

$$
f(\tau) = -\frac{d}{d\tau}\ln D(\tau) = r(\tau) + \tau\, r'(\tau) \ge 0
$$

The check below takes each fitted curve, computes the forward between neighbouring maturities on its grid, and reports the lowest one. A curve passes if that lowest forward is positive.

In [ ]:
rows = []
for method in YieldCurve.METHODS:
    curve = YieldCurve(latest_rates, method)
    forward = curve.curves()["forward_pct"]
    rows.append(
        {
            "curve": curve.name,
            "lowest_forward_pct": round(float(forward.min()), 2),
            "at_years": round(float(forward.idxmin()), 1),
            "arbitrage_free": bool(forward.min() > 0),
        }
    )

arbitrage_check = pd.DataFrame(rows).set_index("curve")
as_of = latest_rates.name.strftime("%Y-%m-%d")
print(f"Arbitrage check ({as_of})")
arbitrage_check


The cell below refits every curve on the last 50 dates in the dataset. For each date it bootstraps that day's par yields to continuous zeros, fits the linear, cubic-spline, Nelson-Siegel, Svensson, Gaussian-process and no-arbitrage Gaussian-process curves, and runs the same arbitrage check as above.

It shows one chart per curve type with a line for each date, a table of the lowest forward rate for each curve and date (any negative value allows arbitrage), and the fitted 10-year zero rate. The full results, with zero rates at every quoted maturity, are in `history`.

In [ ]:
history_dates = df_treasury.index[-50:]
curve_names = list(YieldCurve.METHODS.values())
history_curves = {}
history_rows = []

for date in history_dates:
    par_rates = df_treasury.loc[date]
    for method, name in YieldCurve.METHODS.items():
        curve = YieldCurve(par_rates, method)
        grid = curve.curves()
        history_curves[(date, name)] = grid
        history_rows.append(
            {
                "date": date,
                "curve": name,
                **{
                    label: curve.zero(YieldCurve.MATURITY_YEARS[label])
                    for label in par_rates.index
                },
                "lowest_forward_pct": grid["forward_pct"].min(),
                "arbitrage_free": bool(grid["forward_pct"].min() > 0),
            }
        )

history = pd.DataFrame(history_rows).set_index(["date", "curve"])

colors = plt.cm.viridis(np.linspace(0, 1, len(history_dates)))
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
for ax, name in zip(axes.flat, curve_names):
    passes = history.xs(name, level="curve")["arbitrage_free"]
    for color, date in zip(colors, history_dates):
        grid = history_curves[(date, name)]
        ax.plot(
            grid.index,
            grid["zero_cc_pct"],
            color=color,
            # Dashed lines are dates where this curve allows arbitrage.
            linestyle="-" if passes[date] else "--",
            label=date.strftime("%Y-%m-%d"),
        )
    ax.set_title(f"{name}\narbitrage free on {passes.sum()} of {len(passes)} dates")
    ax.grid(True, alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("Maturity (years)")
for ax in axes[:, 0]:
    ax.set_ylabel("Continuously compounded rate (%)")
fig.suptitle(f"US Treasury Zero Curves, Last {len(history_dates)} Dates")
fig.tight_layout()
# Too many dates for a legend, so a color bar maps line color to date.
date_colors = plt.cm.ScalarMappable(
    cmap="viridis",
    norm=plt.Normalize(0, len(history_dates) - 1),
)
colorbar = fig.colorbar(date_colors, ax=axes, shrink=0.8, pad=0.02)
tick_positions = np.linspace(0, len(history_dates) - 1, 6).round().astype(int)
colorbar.set_ticks(tick_positions)
colorbar.set_ticklabels([history_dates[i].strftime("%Y-%m-%d") for i in tick_positions])
colorbar.set_label("As-of date")
plt.show()

print("Arbitrage free by date and curve")
display(
    history["arbitrage_free"]
    .unstack("curve")[curve_names]
    .replace({True: "Yes", False: "No"})
)
print("Lowest forward rate (%) by date and curve")
display(history["lowest_forward_pct"].unstack("curve")[curve_names].round(2))
print("10-year zero rate (%) by date and curve")
history["10Y"].unstack("curve")[curve_names].round(3)


## Out-of-sample test: leave out 3M and 5Y

A curve that passes through every quote can look perfect in-sample, so here we test how well each method **predicts rates it has not seen**.

For each of the last 50 dates:

1. Drop the 3M and 5Y par yields from the FRED quotes.
2. Bootstrap the remaining 9 par yields to continuous zero rates and build all six curves.
3. Read the estimated zero rates at 3 months and 5 years off each curve.
4. Convert those back to par yields, so they can be compared with the real FRED quotes.

From a continuous zero curve $r(t)$ the discount factor is $D(t) = e^{-r(t)\,t}$. The 3M bill is quoted as a bond-equivalent yield, so

$$
y_{3M} = 2\left(D(0.25)^{-1/(2 \cdot 0.25)} - 1\right).
$$

The 5Y note pays semiannual coupons, and its par yield is the coupon that prices the bond at 100:

$$
y_{5Y} = \frac{2\,\bigl(1 - D(5)\bigr)}{\sum_{i=1}^{10} D(i/2)}.
$$

The error is the estimate minus the real quote, in basis points (1bp = 0.01%). We also compare the estimated zero rate with the zero bootstrapped from the full set of 11 quotes.

In [ ]:
held_out = ["3M", "5Y"]
curve_names = list(YieldCurve.METHODS.values())

holdout_rows = []
for date in history_dates:
    actual_zero = YieldCurve(df_treasury.loc[date]).quotes["yield_cc_pct"]
    kept = df_treasury.loc[date].drop(held_out)
    for method, name in YieldCurve.METHODS.items():
        curve = YieldCurve(kept, method)
        for label in held_out:
            tenor = YieldCurve.MATURITY_YEARS[label]
            est_par = curve.par(tenor)
            est_zero = curve.zero(tenor)
            actual_par = df_treasury.loc[date, label]
            holdout_rows.append(
                {
                    "date": date,
                    "curve": name,
                    "tenor": label,
                    "actual_par_pct": actual_par,
                    "estimated_par_pct": est_par,
                    "par_error_bp": (est_par - actual_par) * 100,
                    "zero_error_bp": (est_zero - actual_zero[label]) * 100,
                }
            )

holdout = pd.DataFrame(holdout_rows)

fig, axes = plt.subplots(1, len(held_out), figsize=(14, 4.5))
for ax, label in zip(axes, held_out):
    rows = holdout[holdout["tenor"] == label]
    actual = rows.drop_duplicates("date").set_index("date")["actual_par_pct"]
    ax.plot(actual.index, actual, color="black", marker="o", linewidth=2.5, label="FRED (actual)")
    for name in curve_names:
        est = rows[rows["curve"] == name].set_index("date")["estimated_par_pct"]
        ax.plot(est.index, est, marker=".", label=name)
    ax.set_title(f"{label} par yield: estimated without the {label} quote vs actual")
    ax.set_ylabel("Par yield (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

summary = holdout.groupby(["tenor", "curve"]).agg(
    mean_error_bp=("par_error_bp", "mean"),
    mean_abs_error_bp=("par_error_bp", lambda e: e.abs().mean()),
    rmse_bp=("par_error_bp", lambda e: np.sqrt((e**2).mean())),
    zero_rmse_bp=("zero_error_bp", lambda e: np.sqrt((e**2).mean())),
)
print("Par yield error by date (bp, estimate minus FRED)")
display(
    holdout.pivot_table(index="date", columns=["tenor", "curve"], values="par_error_bp")
    .reindex(columns=pd.MultiIndex.from_product([held_out, curve_names]))
    .round(1)
)
print(f"Out-of-sample error summary over {len(history_dates)} dates (bp)")
summary.reindex(pd.MultiIndex.from_product([held_out, curve_names])).round(1)


## A `YieldCurve` object

The class is in `curves.py`. You give it one date of FRED par yields (a pandas Series indexed by `"1M"`, `"3M"`, ..., `"30Y"`) and a fitting method. It then:

1. Bootstraps the par yields to continuously compounded zero rates.
2. Fits a zero curve $r(t)$ through them with the chosen method. The default is `"linear"`.
3. Derives the other curves from $r(t)$:

$$
D(t) = e^{-r(t)\,t}, \qquad f(t) = \frac{d}{dt}\bigl(r(t)\,t\bigr) = r(t) + t\,r'(t).
$$

The methods are `"linear"`, `"cubic_spline"`, `"nelson_siegel"`, `"nss"`, `"gp"` and `"no_arb_gp"`. For `"no_arb_gp"` the forward curve is the model's own softplus forward, so it is positive by construction. For the others it is a numerical derivative of $r(t)\,t$.

The main methods are `zero(t)`, `forward(t)`, `discount(t)` and `par(t)`, which all accept a maturity in years or an array of them, plus `curves()`, `is_arbitrage_free()` and `plot()`.


In [ ]:
curve = YieldCurve(latest_rates)
print(curve)
curve.plot()

summary = pd.DataFrame(
    {
        YieldCurve.METHODS[method]: {
            "10Y zero (%)": c.zero(10),
            "10Y forward (%)": c.forward(10),
            "10Y discount factor": c.discount(10),
            "5Y par from curve (%)": c.par(5),
            "arbitrage free": c.is_arbitrage_free(),
        }
        for method in YieldCurve.METHODS
        for c in [YieldCurve(latest_rates, method)]
    }
).T
summary
